In [1]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter      # ganti sesuai provider                     # atau langchain_qdrant, langchain_community.vectorstores.FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


In [14]:
import langchain
import langchain_core

print("langchain:", langchain.__version__)
print("langchain_core:", langchain_core.__version__)

langchain: 0.3.27
langchain_core: 0.3.79


In [2]:
import pandas as pd
data = pd.read_json('datacorpus.json')
data

,id,pasal,bab,judul,context,ayat,buku,bagian,paragraf
0,kuhp_pasal_1,1,BAB I - RUANG LINGKUP BERLAKUNYA KETENTUAN PER...,,Pasal 1\n(1) Tidak ada satu perbuatan pun yang...,"[{'nomor': '1', 'isi': '(1) Tidak ada satu per...",NaN,NaN,NaN
1,kuhp_pasal_2,2,BAB I - RUANG LINGKUP BERLAKUNYA KETENTUAN PER...,,Pasal 2\n(1) Ketentuan sebagaimana dimaksud da...,"[{'nomor': '1', 'isi': '(1) Ketentuan sebagaim...",NaN,NaN,NaN
2,kuhp_pasal_3,3,BAB I - RUANG LINGKUP BERLAKUNYA KETENTUAN PER...,,Pasal 3\n(1) Dalam hal terdapat perubahan pera...,"[{'nomor': '1', 'isi': '(1) Dalam hal terdapat...",NaN,NaN,NaN
3,kuhp_pasal_4,4,BAB I - RUANG LINGKUP BERLAKUNYA KETENTUAN PER...,,Pasal 4\nKetentuan pidana dalam Undang-Undang ...,[],NaN,NaN,NaN
4,kuhp_pasal_5,5,BAB I - RUANG LINGKUP BERLAKUNYA KETENTUAN PER...,,Pasal 5\nKetentuan pidana dalam Undang-Undang ...,[],NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
616,kuhp_pasal_617,617,BAB XXXVI - KETENTUAN PERALIHAN,,Pasal 617\nPada saat Undang-Undang ini mulai b...,[],BUKU KEDUA - TINDAK PIDANA,,NaN
617,kuhp_pasal_618,618,BAB XXXVI - KETENTUAN PERALIHAN,,Pasal 618\nPada saat Undang-Undang ini mulai b...,[],BUKU KEDUA - TINDAK PIDANA,,NaN
618,kuhp_pasal_619,619,BAB XXXVI - KETENTUAN PERALIHAN,,Pasal 619\nPada saat Undang-Undang ini mulai b...,[],BUKU KEDUA - TINDAK PIDANA,,NaN
619,kuhp_pasal_620,620,BAB XXXVI - KETENTUAN PERALIHAN,,Pasal 620\nPada saat Undang-Undang ini mulai b...,[],BUKU KEDUA - TINDAK PIDANA,,NaN


In [3]:
data = data.drop(columns=['id','pasal','bab','judul','ayat','buku','bagian','paragraf'])
data

,context
0,Pasal 1\n(1) Tidak ada satu perbuatan pun yang...
1,Pasal 2\n(1) Ketentuan sebagaimana dimaksud da...
2,Pasal 3\n(1) Dalam hal terdapat perubahan pera...
3,Pasal 4\nKetentuan pidana dalam Undang-Undang ...
4,Pasal 5\nKetentuan pidana dalam Undang-Undang ...
...,...
616,Pasal 617\nPada saat Undang-Undang ini mulai b...
617,Pasal 618\nPada saat Undang-Undang ini mulai b...
618,Pasal 619\nPada saat Undang-Undang ini mulai b...
619,Pasal 620\nPada saat Undang-Undang ini mulai b...


In [4]:
data['context'] = (data['context']
        .str.replace("\n", " ")
        .str.replace(r" +", " ", regex=True)
        .str.strip())
data

,context
0,Pasal 1 (1) Tidak ada satu perbuatan pun yang ...
1,Pasal 2 (1) Ketentuan sebagaimana dimaksud dal...
2,Pasal 3 (1) Dalam hal terdapat perubahan perat...
3,Pasal 4 Ketentuan pidana dalam Undang-Undang b...
4,Pasal 5 Ketentuan pidana dalam Undang-Undang b...
...,...
616,Pasal 617 Pada saat Undang-Undang ini mulai be...
617,Pasal 618 Pada saat Undang-Undang ini mulai be...
618,Pasal 619 Pada saat Undang-Undang ini mulai be...
619,Pasal 620 Pada saat Undang-Undang ini mulai be...


In [5]:
data['context']

0      Pasal 1 (1) Tidak ada satu perbuatan pun yang ...
1      Pasal 2 (1) Ketentuan sebagaimana dimaksud dal...
2      Pasal 3 (1) Dalam hal terdapat perubahan perat...
3      Pasal 4 Ketentuan pidana dalam Undang-Undang b...
4      Pasal 5 Ketentuan pidana dalam Undang-Undang b...
                             ...                        
616    Pasal 617 Pada saat Undang-Undang ini mulai be...
617    Pasal 618 Pada saat Undang-Undang ini mulai be...
618    Pasal 619 Pada saat Undang-Undang ini mulai be...
619    Pasal 620 Pada saat Undang-Undang ini mulai be...
620    Pasal 621 Peraturan pelaksanaan dari Undang-Un...
Name: context, Length: 621, dtype: object

In [6]:
# Chunking — hasilkan list of Document, bukan plain string
splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=50
)

documents = []

for pasal in data['context']:
    if len(pasal) > 4000:
        # split_text → list[str], bungkus jadi Document
        splits = splitter.split_text(pasal)
        documents.extend([Document(page_content=s) for s in splits])
    else:
        documents.append(Document(page_content=pasal))


In [7]:
# Chunking — hasilkan list of Document, bukan plain string
splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=50
)

documents = []

for pasal in data['context']:
    if len(pasal) > 4000:
        # split_text → list[str], bungkus jadi Document
        splits = splitter.split_text(pasal)
        documents.extend([Document(page_content=s) for s in splits])
    else:
        documents.append(Document(page_content=pasal))


In [20]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import os

embeddings = HuggingFaceEmbeddings(
    model_name="mzuama/E5-sampel-law",
    model_kwargs={"device": "cpu"}
)


db_path = "faiss_indexe5lawsampel"

# %%
# Buat atau load FAISS index
if not os.path.exists(db_path):
    vectorstore = FAISS.from_documents(documents, embeddings)  # ✅ Document objects
    vectorstore.save_local(db_path)
else:
    vectorstore = FAISS.load_local(
        db_path,
        embeddings,
        allow_dangerous_deserialization=True
    )


tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/312 [00:00<?, ?B/s]

In [21]:
# ============================================================
# PENDEKATAN 1: Dense Retrieval (FAISS) — sudah kamu punya
# ============================================================
retriever_dense = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}
)

results = retriever_dense.invoke("Apa hukuman untuk orang yang memalsu uang rupiah??")
for i, doc in enumerate(results, 1):
    print(f"[{i}] {doc.page_content[:200]}\n")

[1] Pasal 374 Setiap Orang yang memalsu mata uang atau uang kertas yang dikeluarkan oleh negara, dengan maksud untuk mengedarkan atau meminta mengedarkan sebagai uang asli dan tidak dipalsu, dipidana deng

[2] Pasal 377 Dipidana dengan pidana penjara paling lama 10 (sepuluh) tahun atau pidana denda paling banyak kategori VII, Setiap Orang yang: a. mengedarkan mata uang yang nilainya dikurangi atau mengedark

[3] Pasal 376 Setiap Orang yang mengurangi nilai mata uang dengan maksud untuk mengedarkan atau meminta mengedarkan mata uang yang dikurangi nilainya dipidana karena merusak mata uang, dengan pidana penja

[4] Pasal 378 Setiap Orang yang menerima mata uang atau uang kertas yang dikeluarkan oleh negara yang kemudian diketahui tidak asli, dipalsu atau dirusak, namun tetap mengedarkannya, kecuali yang ditentuk

[5] Pasal 375 (1) Setiap Orang yang menyimpan secara fisik dengan cara apa pun yang diketahuinya merupakan mata uang palsu sebagaimana dimaksud dalam Pasal 374, dipidana dengan